# Modelo C — Experiencias ponderadas por perfil

**Hipótesis:** las coincidencias parciales de tipo y tono de piel aportan información, y la importancia relativa de ambos atributos afecta el ranking.

**Método:** calcular un peso para cada experiencia → rating promedio ponderado por producto → suavizado → Top-10.

No busca un número fijo de vecinos. Una persona que comparte solamente el tono puede aportar evidencia; en el Modelo B no la aportaría si su tipo de piel fuera distinto.

Es tu antiguo Modelo 4, optimizado con estadísticas calculadas exclusivamente sobre entrenamiento.

**Estado:** listo para ejecutar en tu proyecto. Las métricas finales están pendientes; no se incluyen resultados simulados como resultados reales.

## 1. Librerías, datos y split común

Copiá `sephora_common.py` junto a estos notebooks en `notebooks/`. Ese archivo contiene las reglas compartidas de preparación, partición y evaluación. El algoritmo particular de este modelo aparece completo abajo.

Se leen `data/processed/reviews_clean.parquet` y `products_clean.parquet`, con alternativa CSV si no existe el parquet. Ejecutar desde arriba con un kernel que tenga las dependencias instaladas.

In [1]:
from pathlib import Path
import sys
import heapq
from itertools import islice
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# El archivo sephora_common.py debe estar junto a los notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrí este notebook dentro del proyecto que contiene data/processed")
helper_dir = next((p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
                   if (p / "sephora_common.py").exists()), None)
if helper_dir is None:
    raise FileNotFoundError("Copiá sephora_common.py a notebooks/ junto a estos archivos")
sys.path.insert(0, str(helper_dir))
import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores, text_value

# Los mismos valores en los cuatro notebooks.
N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42
ctx = sc.load_context(N_VALIDACION, N_TEST, SEMILLA)
print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)
display(pd.Series({k:v for k,v in ctx.manifest.items()
                   if k not in ["validation_cases", "test_cases", "versions"]}).to_frame("valor"))
display(pd.DataFrame(ctx.manifest["versions"], index=[0]))

Protocolo: 99fc64b0ea63
Resultados: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


,valor
version,cold-start-users-v2.0
seed,42
protocol_id,99fc64b0ea63f0e1e20f6a8a2d96aee1222e401c635a55...
reviews_input,1093855
reviews_valid_before_pair_dedup,1093855
reviews_unique_pairs,1088886
removed_duplicate_pairs,4969
catalog_skincare,2420
users_total,503216
eligible_users,161396


,python,numpy,pandas,scipy,scikit_learn
0,3.14.5,2.4.6,3.0.3,1.18.1,1.9.0


## 2. Condiciones comunes de comparación

- **Consulta:** tipo y tono de piel, ojos y cabello. Cada enfoque decide qué atributos utiliza. Ninguno recibe compras ni ratings del usuario de consulta.
- **Candidatos:** catálogo Skincare, idéntico para todos. Un modelo basado en evidencia puede puntuar solo parte de este catálogo; ese límite se refleja en la cobertura.
- **Positivo:** `rating >= 4` **o** `is_recommended = True`. La misma regla sirve para todos. Una review de rating bajo pero recomendada es positiva según esta definición.
- **Datos:** un par usuario–producto por vez; se conserva la última review con fecha disponible. Se cuantifica lo descartado. Los perfiles se resumen por moda, sin utilizar ratings para definirlos.
- **Validación:** 500 usuarios para elegir hiperparámetros.
- **Test:** otros 500 usuarios para evaluar la configuración elegida. Se fija un producto positivo por usuario.
- **Sin fuga:** ninguna review de esos 1.000 usuarios entra en entrenamiento, estadísticas de ratings, popularidad o votos.
- **Cold start:** los demás positivos del usuario también permanecen ocultos. No se utilizan ni siquiera para filtrar el Top-10.

Se seleccionan usuarios con al menos dos productos positivos y tipo/tono conocidos. Es una evaluación en esa población, no en todos los visitantes del sitio. Se supone que los atributos personales aportados en las reviews son conocidos al consultar. Esta partición no simula el paso del tiempo.

La huella del protocolo verifica que datos, semilla y tamaños sean iguales. Si cambian, los resultados van a otra carpeta; el comparador no mezcla protocolos.

## 3. Cómo se calcula la similitud y el score

$$w = a\,I(tipo\ igual) + b\,I(tono\ igual), \quad a+b=1$$

Con `a=0,7` y `b=0,3`: coincidir en ambas características pesa 1; solo en tipo pesa 0,7; solo en tono pesa 0,3; en ninguna pesa 0.

Por producto, la evidencia ponderada es `E = suma de pesos`; el rating ponderado es `R = suma(w × rating) / E`. El score se suaviza:

$$score = \frac{E}{E+m}R + \frac{m}{E+m}C$$

`C` es la media ponderada de las experiencias admitidas para ese perfil. `m=50` se fija como referencia; no se afirma que sea óptimo. Se ajustan los pesos en validación.

Se usan perfiles con tipo y tono conocidos. Un grupo puede aportar muchas reviews, pero el peso depende de las coincidencias. El precálculo no contiene a los usuarios de validación ni test.

**Limitación:** sigue suponiendo que tipo y tono explican preferencias; no aprende automáticamente esa relación. Comparte la idea de suavizado con B, pero cambia quién aporta evidencia y cuánto pesa.

In [2]:
class WeightedProfileSatisfaction:
    """Cada experiencia pesa según coincidencia de tipo y tono de piel."""
    def __init__(self, ctx):
        r = ctx.train[ctx.train.skin_type.ne("Unknown") & ctx.train.skin_tone.ne("Unknown")]
        self.stats = r.groupby(["skin_type", "skin_tone", "product_id"]).rating.agg(["sum", "count"]).reset_index()
        self.cache = {}

    def scores(self, profile, config):
        wt, wn, m = float(config["weight_type"]), float(config["weight_tone"]), float(config["m"])
        if wt < 0 or wn < 0 or not np.isclose(wt+wn, 1):
            raise ValueError("Los pesos deben ser no negativos y sumar 1")
        key = (profile.get("skin_type"), profile.get("skin_tone"), wt, wn, m)
        if key in self.cache:
            return self.cache[key]
        d = self.stats.copy()
        weight = wt*d.skin_type.eq(key[0]).astype(float) + wn*d.skin_tone.eq(key[1]).astype(float)
        d["weighted_sum"] = d["sum"]*weight
        d["weighted_count"] = d["count"]*weight
        d = d[weight > 0]
        grouped = d.groupby("product_id")[["weighted_sum", "weighted_count"]].sum().reset_index()
        if not grouped.empty:
            prior = grouped.weighted_sum.sum()/grouped.weighted_count.sum()
            grouped["score"] = (grouped.weighted_sum + m*prior)/(grouped.weighted_count+m)
        self.cache[key] = grouped
        return grouped

    def recommend(self, profile, config, top_n=10, exclude=()):
        return rank_scores(self.scores(profile, config), "score", top_n, exclude)

## 4. Preparación del modelo y configuraciones

In [3]:
model = WeightedProfileSatisfaction(ctx)
CONFIGURACIONES = [{"weight_type": a, "weight_tone": b, "m": 50}
                   for a, b in [(1.0, 0.0), (0.8, 0.2), (0.7, 0.3), (0.5, 0.5)]]
print("Combinaciones tipo–tono–producto:", len(model.stats))

Combinaciones tipo–tono–producto: 53519


## 5. Ajuste en validación y evaluación final

**Hit Rate@10:** usuarios cuyo producto oculto aparece en el Top-10 / usuarios evaluados.

**NDCG@10:** vale 0 si no aparece; si aparece en posición `r`, vale `1/log2(r+1)`. Con un solo positivo de prueba por usuario, premia que el acierto esté más arriba.

**Cobertura de usuarios:** porcentaje que recibe al menos una recomendación. **Top-10 completo:** porcentaje que recibe diez. **Cobertura de catálogo:** productos distintos recomendados / productos de skincare del catálogo.

Los usuarios sin recomendaciones cuentan como misses. No se eliminan del denominador. El parámetro se elige por HR@10 de validación y, en empate, por NDCG@10. Un empate total conserva la primera configuración de la lista. Test se utiliza después de esa elección.

Se compara con popularidad positiva global calculada exclusivamente sobre train. El promedio de tiempo por consulta es orientativo: puede incluir efectos de caché y no constituye un benchmark de producción.

In [4]:
mejor_config, validacion, metricas_test, detalle_test, baseline_test = sc.run_experiment(
    model, ctx, CONFIGURACIONES)
print("Resultados de validación:")
display(validacion)
print("Configuración elegida:", mejor_config)
print("Resultados finales en test:")
display(pd.DataFrame([metricas_test, baseline_test], index=["Modelo", "Baseline global"]))

Validación: {'weight_type': 1.0, 'weight_tone': 0.0, 'm': 50}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'weight_type': 0.8, 'weight_tone': 0.2, 'm': 50}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'weight_type': 0.7, 'weight_tone': 0.3, 'm': 50}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Validación: {'weight_type': 0.5, 'weight_tone': 0.5, 'm': 50}
Evaluados 50/500
Evaluados 100/500
Evaluados 150/500
Evaluados 200/500
Evaluados 250/500
Evaluados 300/500
Evaluados 350/500
Evaluados 400/500
Evaluados 450/500
Evaluados 500/500
Configuración seleccionada: {'weight

,config,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
0,"{'weight_type': 1.0, 'weight_tone': 0.0, 'm': 50}",500,3,0.006,0.002614,1.0,1.0,0.008264,10.0,0.009394
1,"{'weight_type': 0.8, 'weight_tone': 0.2, 'm': 50}",500,3,0.006,0.002893,1.0,1.0,0.009504,10.0,0.009137
2,"{'weight_type': 0.7, 'weight_tone': 0.3, 'm': 50}",500,4,0.008,0.003507,1.0,1.0,0.009917,10.0,0.014321
3,"{'weight_type': 0.5, 'weight_tone': 0.5, 'm': 50}",500,4,0.008,0.003748,1.0,1.0,0.012397,10.0,0.012890


Configuración elegida: {'weight_type': 0.5, 'weight_tone': 0.5, 'm': 50}
Resultados finales en test:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
Modelo,500,8,0.016,0.006803,1.0,1.0,0.013636,10.0,0.010227
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.008454


## 6. Ejemplo de Top-10 para una persona nueva

Todos los notebooks usan el mismo perfil de ejemplo. Acá no se oculta un producto ni se mide un acierto: se muestra qué devolvería la consulta real. Si no hay evidencia suficiente, la tabla puede tener menos de diez filas.

El ejemplo no reemplaza a la evaluación agregada. No significa que estos productos sean clínicamente adecuados para esa persona.

In [5]:
PERFIL_EJEMPLO = {"skin_type": "oily", "skin_tone": "dark", "eye_color": "blue", "hair_color": "brown"}
perfil, top10 = sc.example_top10(model, ctx, mejor_config, PERFIL_EJEMPLO)
display(pd.DataFrame([perfil]))
display(top10)
print("Productos devueltos:", len(top10))

,skin_type,skin_tone,eye_color,hair_color
0,oily,dark,blue,brown


,position,product_id,product_name,brand_name,secondary_category,price_usd
0,1,P423688,Daily Microfoliant Exfoliator,Dermalogica,Cleansers,65.0
1,2,P443352,Mini Daily Microfoliant Exfoliator,Dermalogica,Mini Size,18.0
2,3,P173726,Facial Cotton,Shiseido,Cleansers,13.0
3,4,P42204,Rosebud Salve,Rosebud Perfume Co.,Lip Balms & Treatments,7.0
4,5,P479841,Floral Recovery Overnight Mask with Squalane,fresh,Masks,69.0
5,6,P442563,AHA 30% + BHA 2% Exfoliating Peeling Solution,The Ordinary,Treatments,9.5
6,7,P500633,Tea Elixir Niacinamide & Hyaluronic Acid Anti-...,fresh,Treatments,80.0
7,8,P456412,Greek Yoghurt Foaming Cream Cleanser,KORRES,Cleansers,28.0
8,9,P397890,Original Skin Retexturizing Mask with Rose Clay,Origins,Masks,32.0
9,10,P12573,Intense Therapy Lip Balm SPF 25,Jack Black,Lip Balms & Treatments,10.0


Productos devueltos: 10


## 7. Cierre y exportación de evidencia

La conclusión se calcula con tus resultados nuevos. No reutilizar el 6,2 %, 5,6 %, 2,6 % o 1,4 % de los notebooks anteriores: cambiaron el split y las exclusiones.

Analizar: ¿supera al baseline? ¿Cuánto catálogo cubre? ¿Entrega diez productos? ¿Qué información del perfil usa? ¿Qué sesgos mantiene? Un resultado bajo es evidencia útil, no un motivo para ocultar el experimento.

In [6]:
sc.conclusion(metricas_test, baseline_test)
extra = {"algorithm": "WeightedProfileSatisfaction"}

archivo = sc.save_result(ctx, "03_similitud_ponderada", mejor_config, validacion, metricas_test,
                         detalle_test, baseline_test, top10, extra)
print("Resultado guardado:", archivo)

HR@10 test: 1.60%; baseline: 7.40%; diferencia: -5.80 puntos porcentuales.
Cobertura de usuarios: 100.00%; catálogo: 1.36%.
En esta muestra no supera al baseline en HR. No declararlo ganador por personalización solamente.
Un hit recupera una preferencia observada; no prueba eficacia ni compatibilidad dermatológica.
Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63\03_similitud_ponderada.json


## 8. Preguntas para entenderlo y defenderlo

1. ¿Qué información usa el algoritmo para ordenar productos?
2. ¿Qué parámetro ajustamos y por qué?
3. ¿Cómo se asegura que el usuario de test no aporte evidencia al ranking?
4. ¿Qué significa un hit y qué no demuestra?
5. ¿Qué aprendimos al compararlo con popularidad global?

Guardá con **Ctrl + S** para conservar todas las salidas y abrí luego `05_comparacion_final.ipynb`.